# 04 - SOH / capacity prediction si RUL derivat

Acest notebook incearca varianta folosita des in literatura: in loc sa prezicem direct RUL, prezicem o marime mai lina, **SOH/capacity ratio**, apoi estimam RUL fata de un prag EOL.

Important:

- Nu folosim `capacity_ah_clean` sau `capacity_ratio` ca input direct pentru acelasi ciclu, fiindca ar fi leakage.
- Folosim feature-uri de semnal masurat si feature-uri istorice lagged, calculate din ciclurile anterioare.
- Rulam pe aceleasi scenarii ca notebookurile 02/03: `all_eligible`, `clean_benchmark`, `nasa_classic_4`.

## 1. Imports si paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Could not find the number of physical cores")

SEED = 42
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

ARTIFACTS_DIR = REPO_ROOT / "artifacts"
FEATURE_TABLE_PATH = ARTIFACTS_DIR / "features" / "battery_cycle_features_v2.csv"
FEATURE_COLUMNS_PATH = ARTIFACTS_DIR / "features" / "baseline_feature_columns_v2.json"
SCENARIO_CONFIG_PATH = ARTIFACTS_DIR / "splits" / "modeling_scenarios_v1.json"
MODEL_DIR = ARTIFACTS_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"
PRED_DIR = ARTIFACTS_DIR / "predictions"
FIG_DIR = ARTIFACTS_DIR / "figures" / "soh_capacity"
for d in [MODEL_DIR, METRICS_DIR, PRED_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)

assert FEATURE_TABLE_PATH.exists(), "Run notebook 02 first."
assert FEATURE_COLUMNS_PATH.exists(), "Run notebook 02 first."
assert SCENARIO_CONFIG_PATH.exists(), "Run notebook 02 first."

## 2. Incarcare date si feature-uri lagged

In [ ]:
model_df = pd.read_csv(FEATURE_TABLE_PATH, parse_dates=["start_time", "imp_start_time"])
feature_config = json.loads(FEATURE_COLUMNS_PATH.read_text())
scenarios = json.loads(SCENARIO_CONFIG_PATH.read_text())
base_feature_cols = feature_config["feature_cols"]
GROUP_COL = feature_config["group_col"]
TARGET_SOH = "soh"
SCENARIOS_TO_RUN = ["all_eligible", "clean_benchmark", "nasa_classic_4"]

def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(["battery_id", "cycle_index"]).copy()
    for battery_id, group in out.groupby("battery_id"):
        idx = group.sort_values("cycle_index").index
        soh = out.loc[idx, "soh"].astype(float)
        # Use only information available before the current cycle. The first
        # cycle falls back to nominal SOH=1.0 instead of backfilling from future rows.
        prev_soh = soh.shift(1).fillna(1.0)
        out.loc[idx, "prev_soh"] = prev_soh
        out.loc[idx, "prev_soh_delta1"] = soh.diff().shift(1).fillna(0.0)
        out.loc[idx, "prev_soh_rollmean5"] = prev_soh.rolling(5, min_periods=1).mean().fillna(1.0)
        out.loc[idx, "prev_soh_rollstd5"] = prev_soh.rolling(5, min_periods=2).std().fillna(0.0)
        out.loc[idx, "prev_soh_slope5"] = prev_soh.rolling(5, min_periods=3).apply(
            lambda x: np.polyfit(np.arange(len(x)), x, 1)[0] if len(x) >= 3 else 0.0,
            raw=False,
        ).fillna(0.0)
    return out

model_df = add_lag_features(model_df)

# Exclude current-cycle capacity/SOH-derived features to avoid target leakage.
leaky_features = {
    "capacity_ah_clean",
    "capacity_ratio",
    "cap_ratio_delta1",
    "cap_ratio_rollmean5",
    "cap_ratio_rollstd5",
    "cap_ratio_slope5",
}
soh_feature_cols = [c for c in base_feature_cols if c not in leaky_features]
soh_feature_cols += ["prev_soh", "prev_soh_delta1", "prev_soh_rollmean5", "prev_soh_rollstd5", "prev_soh_slope5"]
soh_feature_cols = [c for c in soh_feature_cols if c in model_df.columns]

feature_path = ARTIFACTS_DIR / "features" / "soh_feature_columns_v2.json"
feature_path.write_text(json.dumps({"target_col": TARGET_SOH, "feature_cols": soh_feature_cols, "leaky_features_excluded": sorted(leaky_features)}, indent=2))
print("SOH feature count:", len(soh_feature_cols))
print(soh_feature_cols)

## 3. Modele si metrici

In [ ]:
def regression_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def make_scaled_model(model):
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", model)])


def make_tree_model(model):
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", model)])

models = {
    "SVR_RBF": make_scaled_model(SVR(kernel="rbf", C=10.0, epsilon=0.005, gamma="scale")),
    "RandomForest": make_tree_model(RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=SEED, n_jobs=1)),
    "ExtraTrees": make_tree_model(ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=SEED, n_jobs=1)),
    "HistGradientBoosting": make_tree_model(HistGradientBoostingRegressor(max_iter=500, learning_rate=0.03, l2_regularization=0.02, random_state=SEED)),
    "GradientBoosting": make_tree_model(GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=2, random_state=SEED)),
}

## 4. Training/evaluare SOH pe scenarii

In [ ]:
validation_rows = []
test_rows = []
prediction_frames = []
summary = {}

for scenario_name in SCENARIOS_TO_RUN:
    print("\n" + "=" * 80)
    print("Scenario:", scenario_name)
    cfg = scenarios[scenario_name]
    split = cfg["split"]
    scenario_df = model_df[model_df[GROUP_COL].isin(cfg["battery_ids"])].copy()
    train_df = scenario_df[scenario_df[GROUP_COL].isin(split["train_batteries"])].copy()
    val_df = scenario_df[scenario_df[GROUP_COL].isin(split["validation_batteries"])].copy()
    test_df = scenario_df[scenario_df[GROUP_COL].isin(split["test_batteries"])].copy()
    trainval_df = scenario_df[scenario_df[GROUP_COL].isin(split["train_batteries"] + split["validation_batteries"])].copy()

    print("train/val/test rows:", len(train_df), len(val_df), len(test_df))
    val_metrics = []
    for model_name, estimator in models.items():
        model = clone(estimator)
        model.fit(train_df[soh_feature_cols], train_df[TARGET_SOH])
        pred_val = np.clip(model.predict(val_df[soh_feature_cols]), 0.0, 1.30)
        val_metrics.append({"scenario": scenario_name, "model": model_name, "split": "validation", **regression_metrics(val_df[TARGET_SOH], pred_val)})

    val_df_metrics = pd.DataFrame(val_metrics).sort_values("RMSE")
    validation_rows.extend(val_metrics)
    best_validation_model = val_df_metrics.iloc[0]["model"]
    print("Best validation model:", best_validation_model)

    for model_name, estimator in models.items():
        model = clone(estimator)
        model.fit(trainval_df[soh_feature_cols], trainval_df[TARGET_SOH])
        pred_test = np.clip(model.predict(test_df[soh_feature_cols]), 0.0, 1.30)
        test_rows.append({"scenario": scenario_name, "model": model_name, "split": "test", **regression_metrics(test_df[TARGET_SOH], pred_test)})

        pred_frame = test_df[["battery_id", "cycle_index", "start_time", "soh", "capacity_ah_clean", "rul_cycles"]].copy()
        pred_frame["scenario"] = scenario_name
        pred_frame["model"] = model_name
        pred_frame["pred_soh"] = pred_test
        pred_frame["soh_error"] = pred_frame["pred_soh"] - pred_frame["soh"]
        pred_frame["abs_soh_error"] = pred_frame["soh_error"].abs()
        prediction_frames.append(pred_frame)

        model_path = MODEL_DIR / f"soh_{scenario_name}_{model_name.lower()}_trainval.joblib"
        joblib.dump(model, model_path)

    summary[scenario_name] = {"best_validation_model": best_validation_model, "split": split}

validation_metrics = pd.DataFrame(validation_rows).sort_values(["scenario", "RMSE"])
test_metrics = pd.DataFrame(test_rows).sort_values(["scenario", "RMSE"])
predictions = pd.concat(prediction_frames, ignore_index=True)

validation_metrics.to_csv(METRICS_DIR / "soh_validation_metrics.csv", index=False)
test_metrics.to_csv(METRICS_DIR / "soh_test_metrics.csv", index=False)
predictions.to_csv(PRED_DIR / "soh_test_predictions.csv", index=False)
(METRICS_DIR / "soh_experiment_summary.json").write_text(json.dumps(summary, indent=2))

display(validation_metrics)
display(test_metrics)

## 5. Grafice SOH/capacity prediction

In [ ]:
for scenario_name in SCENARIOS_TO_RUN:
    scenario_metrics = test_metrics[test_metrics["scenario"] == scenario_name].sort_values("RMSE")
    best_test_model = scenario_metrics.iloc[0]["model"]
    selected = predictions[(predictions["scenario"] == scenario_name) & (predictions["model"] == best_test_model)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(selected["soh"], selected["pred_soh"], s=18, alpha=0.6)
    line_min = float(min(selected["soh"].min(), selected["pred_soh"].min()))
    line_max = float(max(selected["soh"].max(), selected["pred_soh"].max()))
    axes[0].plot([line_min, line_max], [line_min, line_max], color="red", linestyle="--", linewidth=1)
    axes[0].set_title(f"{scenario_name}: {best_test_model} SOH predicted vs true")
    axes[0].set_xlabel("True SOH")
    axes[0].set_ylabel("Predicted SOH")
    axes[1].hist(selected["soh_error"], bins=35, color="#3a6ea5")
    axes[1].axvline(0, color="red", linestyle="--", linewidth=1)
    axes[1].set_title("SOH prediction error")
    axes[1].set_xlabel("Predicted - true SOH")
    plt.tight_layout()
    fig_path = FIG_DIR / f"01_{scenario_name}_best_soh_pred_vs_true.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()

    test_battery_ids = sorted(selected["battery_id"].unique())
    cols = 2
    rows_n = int(np.ceil(len(test_battery_ids) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(14, 4 * rows_n), squeeze=False)
    for ax, battery_id in zip(axes.ravel(), test_battery_ids):
        g = selected[selected["battery_id"] == battery_id].sort_values("cycle_index")
        ax.plot(g["cycle_index"], g["soh"], label="true SOH", linewidth=2)
        ax.plot(g["cycle_index"], g["pred_soh"], label="predicted SOH", linewidth=1.6)
        ax.axhline(0.80, color="red", linestyle="--", linewidth=1, label="80% EOL" if battery_id == test_battery_ids[0] else None)
        ax.set_title(battery_id)
        ax.set_xlabel("Cycle index")
        ax.set_ylabel("SOH")
        ax.legend(fontsize=8)
    for ax in axes.ravel()[len(test_battery_ids):]:
        ax.axis("off")
    plt.tight_layout()
    fig_path = FIG_DIR / f"02_{scenario_name}_best_soh_curves_by_battery.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()

## 6. RUL derivat din prag EOL 80%

In [ ]:
EOL_THRESHOLD = 0.80
rul_rows = []

for (scenario_name, model_name, battery_id), g in predictions.groupby(["scenario", "model", "battery_id"]):
    g = g.sort_values("cycle_index").copy()
    true_cross = g.loc[g["soh"] <= EOL_THRESHOLD, "cycle_index"]
    pred_cross = g.loc[g["pred_soh"] <= EOL_THRESHOLD, "cycle_index"]
    if true_cross.empty or pred_cross.empty:
        continue
    true_eol_cycle = int(true_cross.iloc[0])
    pred_eol_cycle = int(pred_cross.iloc[0])
    g["true_rul_to_80"] = (true_eol_cycle - g["cycle_index"]).clip(lower=0)
    g["pred_rul_to_80"] = (pred_eol_cycle - g["cycle_index"]).clip(lower=0)
    metrics = regression_metrics(g["true_rul_to_80"], g["pred_rul_to_80"])
    rul_rows.append(
        {
            "scenario": scenario_name,
            "model": model_name,
            "battery_id": battery_id,
            "true_eol_cycle_80": true_eol_cycle,
            "pred_eol_cycle_80": pred_eol_cycle,
            "eol_cycle_error": pred_eol_cycle - true_eol_cycle,
            **metrics,
        }
    )

rul_derived_metrics = pd.DataFrame(rul_rows).sort_values(["scenario", "model", "battery_id"])
rul_derived_metrics.to_csv(METRICS_DIR / "soh_derived_rul_to_80_metrics.csv", index=False)
display(rul_derived_metrics)

## 7. Interpretare

Acest experiment este cel mai potrivit pentru rezultate vizuale bune:

- SOH este mai lin decat RUL direct;
- putem raporta R2 pe SOH/capacity prediction;
- RUL poate fi derivat apoi fata de un prag EOL, dar derivarea trebuie explicata separat.

Pentru lucrare, o prezentare solida este:

1. RUL direct: benchmark ML + CNN-LSTM;
2. SOH prediction: rezultat mai stabil si mai aproape de literatura;
3. RUL derivat: aplicatie practica pentru demo.